# Part 2 Starter Notebook

This notebook loads an AIME 2024 dataset, runs a model on each problem, extracts an AIME-style final answer, and grades the outputs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import re
import os

import pandas as pd
import torch
from datasets import load_dataset
# from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
from google.colab import userdata

print("Token set" if userdata.get('HF_TOKEN') else "Token not set")

In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B"
# or allenai/Olmo-3-7B-Think
# MODEL_NAME = "allenai/Olmo-3-7B-Think"
DATASET_NAME = "OpenRLHF/aime-2024"
# MAX_NEW_TOKENS = 8000
MAX_NEW_TOKENS = 16000

## Loading the model and the data

In [ ]:
# device = "cuda" if torch.cuda.is_available() else "cpu"
# # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# hf_token = os.environ.get("HF_TOKEN")

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     dtype=torch.float16 if device == "cuda" else torch.float32,
#     token=hf_token
# )
# model.to(device)
# print(next(model.parameters()).device)

# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "left"

# dataset = load_dataset(DATASET_NAME, split="train", token=hf_token)

In [ ]:
print(tokenizer.padding_side)

## Evaluation helpers

In [ ]:
def strip_thinking_trace(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<\|begin_of_thought\|>.*?<\|end_of_thought\|>", "", text, flags=re.DOTALL)
    return text.strip()



def extract_answer(text: str, mode="exact_match") -> int | None:
    """Extract an AIME-style integer answer from a model completion."""
    answer_text = strip_thinking_trace(text)
    if not answer_text:
        if mode == "exact_match":
            return None
        else:
            answer_text = text  # fall back to full text


    # 1. Boxed LaTeX answer: \boxed{123}
    if mode == "exact_match":
        boxed = re.findall(r"\\boxed\{(\d+)\}", answer_text)
        if boxed:
            val = int(boxed[-1])
            return val
        else:
            return None

    elif mode == "flexible_extract":
        # 2. "The answer is N" or "answer: N" patterns
        patterns = [
            r"(?:the\s+)?answer\s+is\s+[:\s]*(\d+)",
            r"answer[:\s]+(\d+)",
            r"=\s*(\d+)\s*$",
            r"(?:therefore|thus|so),?\s+(\d+)\s*(?:\.|$)",
        ]
        for pattern in patterns:
            matches = re.findall(pattern, answer_text, re.IGNORECASE)
            if matches:
                val = int(matches[-1])
                return val

        # 3. Last integer in [0, 999] in the answer portion
        integers = re.findall(r"\b(\d{1,3})\b", answer_text)
        for candidate in reversed(integers):
            val = int(candidate)
            return val
        return None


## Inference

You can also explore using vLLM to speed up inference!

For sequential scaling (token-by-token generation) --> need generation that:
- monitors token stream incrementally
- detects thought boundaries
- intervenes dynamically

For WAIT intervention:
- detect end-think token before budget reached
- override generation
- continue chain

For parallel scaling:
- requires stochastic sampling
- multiple completions
- currently only evaluate single completion --> need answer = [multiple samples] then aggregation logic

Plots needed:
- accuracy vs thinking tokens
- total tokens vs accuracy
- error bars
- exact vs flexible curves

## Warmup (part 2.1)

Run diagnostics first (to see if output is as expected)

In [ ]:
# ENABLE_THINKING = True  # Set False for no-thinking condition
# ANSWER_MODE = "exact_match"

model.to(device)
model.eval()

BATCH_SIZE = 4
records = []

example = dataset[0]
prompt = tokenizer.apply_chat_template(
    example["prompt"], tokenize=False,
    add_generation_prompt=True, enable_thinking=True,
)
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=8000, do_sample=False,
                         temperature=0.6, top_p=0.95, repetition_penalty=1.5)

text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(repr(text[:500])) # first 500 chars
print("...")
print(repr(text[-200:]))  # last 200 chars
print(f"\nTotal tokens generated: {len(out[0]) - inputs['input_ids'].shape[1]}")
print(f"Contains </think>: {'</think>' in text}")

Using vLLM (batch inference)

In [ ]:
!pip install vllm

In [ ]:
import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

from vllm import LLM, SamplingParams
from datasets import load_dataset
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

# from run_eval import extract_answer

BATCH_SIZE = 8
THINK_BUDGET = 16000
ANSWER_TOKENS = 2048    # answer tokens, for output without thinking
ANSWER_MODE = "exact_match"

hf_token = os.environ.get("HF_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

dataset = load_dataset(DATASET_NAME, split="train", token=hf_token)

llm = LLM(model=MODEL_NAME)

params_no_think = SamplingParams(
    max_tokens=ANSWER_TOKENS,
    temperature=0.0,
    top_p=1.0,
)
params_think = SamplingParams(
    max_tokens=THINK_BUDGET + ANSWER_TOKENS,
    temperature=0.0,
    top_p=1.0,
)

In [ ]:
# no thinking eval
records_no_think = []

for i in range(0, len(dataset), BATCH_SIZE):
    batch = dataset.select(range(i, min(i + BATCH_SIZE, len(dataset))))
    prompts, labels = [], []

    for ex in batch:
        prompt = tokenizer.apply_chat_template(
            ex["prompt"], tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        prompts.append(prompt)
        labels.append(int(ex["label"]))

    outputs = llm.generate(prompts, params_no_think)

    for j, out in enumerate(outputs):
        pred = extract_answer(out.outputs[0].text, mode=ANSWER_MODE)
        gold = labels[j]
        records_no_think.append(pred == gold if pred is not None else False)

# thinking eval
records_think = []
think_lengths  = []

for i in range(0, len(dataset), BATCH_SIZE):
    batch = dataset.select(range(i, min(i + BATCH_SIZE, len(dataset))))
    prompts, labels = [], []

    for ex in batch:
        prompt = tokenizer.apply_chat_template(
            ex["prompt"], tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
        prompts.append(prompt)
        labels.append(int(ex["label"]))

    outputs = llm.generate(prompts, params_think)

    for j, out in enumerate(outputs):
        text = out.outputs[0].text
        gold = labels[j]

        think_len = 0
        start = text.find("<think>")
        end   = text.find("</think>")

        if start != -1:
            think_text = text[start + len("<think>"): end if end != -1 else None]
            if think_text:
                think_len = len(
                    tokenizer(think_text, add_special_tokens=False)["input_ids"]
                )

        think_lengths.append(think_len)

        pred = extract_answer(text, mode=ANSWER_MODE)
        records_think.append(pred == gold if pred is not None else False)

acc_no_think = sum(records_no_think) / len(records_no_think)
acc_think    = sum(records_think)    / len(records_think)
avg_think_len = sum(think_lengths)   / len(think_lengths)

print(f"Accuracy (no thinking): {acc_no_think:.4f}")
print(f"Accuracy (with thinking): {acc_think:.4f}")
print(f"Avg think length: {avg_think_len:.1f} tokens")
print(f"Problems with no <think> tag: {think_lengths.count(0)}/30")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(
    [l for l in think_lengths if l > 0],
    bins=20, color="#2563eb", edgecolor="white", linewidth=0.5,
)
ax.axvline(avg_think_len, color="#dc2626", linestyle="--",
           linewidth=1.5, label=f"mean = {avg_think_len:.0f}")
ax.set_xlabel("Thinking length (tokens)")
ax.set_ylabel("Count")
ax.set_title("Thinking length distribution (greedy, with thinking)")
ax.legend()
plt.tight_layout()
plt.savefig("warmup_think_length_hist.png", dpi=150, bbox_inches="tight")
plt.show()

# sanity check
assert len(records_no_think) == len(dataset), "no-think records length mismatch"
assert len(records_think) == len(dataset), "think records length mismatch"
assert len(think_lengths) == len(dataset), "think_lengths length mismatch"
print("All length checks passed.")

## Scaling experiments (part 2.2)

In [ ]:
!python run_eval.py

Plot scaling experiments

In [ ]:
import numpy as np
import matplotlib.ticker as ticker

# hard coded from run logs

# sequential: (budget, exact, flex, total_think_toks, avg_think_per_problem)
seq_results = [
    (1024, 0.1000, 0.1333,  30_720,  1024.0),
    (4000, 0.2333, 0.3000, 118_329,  3944.3),
    (16000, 0.6333, 0.6333, 330_933, 11031.1),
    (32000, 0.7000, 0.7000, 414_749, 13825.0),
]

# parallel: (n, majority, best_of_n, total_think_toks, avg_think_per_problem)
par_results = [
    (1, 0.1667, 0.1667, 172_749,  5758.3),
    (2, 0.2667, 0.2667, 339_307, 11310.2),
    (4, 0.2667, 0.2667, 696_547, 23218.2),
    (8, 0.4000, 0.4000, 1_392_257, 46408.6),
]

seq_budgets = [r[0] for r in seq_results]
seq_exact = [r[1] for r in seq_results]
seq_flex = [r[2] for r in seq_results]
seq_think_toks = [r[3] for r in seq_results]
seq_avg_think = [r[4] for r in seq_results]

par_n = [r[0] for r in par_results]
par_majority = [r[1] for r in par_results]
par_best = [r[2] for r in par_results]
par_think_toks = [r[3] for r in par_results]
par_avg_think = [r[4] for r in par_results]

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": "--",
})

SEQ_EXACT_COLOR = "#2563eb"
SEQ_FLEX_COLOR = "#93c5fd"
PAR_MAJ_COLOR = "#16a34a"
PAR_BEST_COLOR = "#86efac"

fmt = ticker.FuncFormatter(
    lambda v, _: f"{int(v/1000)}K" if v >= 1000 else str(int(v))
)

# plot 1: sequential
fig, ax = plt.subplots(figsize=(8, 5))
ax.set_title("Sequential scaling — thinking tokens vs accuracy (AIME-2024)",
             fontsize=12, fontweight="bold")

ax.plot(seq_think_toks, seq_exact, "o-",  color=SEQ_EXACT_COLOR,
        lw=2, markersize=7, label="exact_match")
ax.plot(seq_think_toks, seq_flex,  "s--", color=SEQ_FLEX_COLOR,
        lw=2, markersize=7, label="flexible_extract")

for xi, ye, yf in zip(seq_think_toks, seq_exact, seq_flex):
    ax.annotate(f"{ye:.2f}", (xi, ye), textcoords="offset points",
                xytext=(0, 9),  ha="center", fontsize=9, color=SEQ_EXACT_COLOR)
    ax.annotate(f"{yf:.2f}", (xi, yf), textcoords="offset points",
                xytext=(0, -15), ha="center", fontsize=9, color=SEQ_FLEX_COLOR)

ax.set_xlabel("Total thinking tokens (30 problems)")
ax.set_ylabel("Accuracy")
ax.xaxis.set_major_formatter(fmt)
ax.set_ylim(-0.02, 1.0)
ax.legend()
plt.tight_layout()
plt.savefig("seq_thinking_tokens_vs_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: seq_thinking_tokens_vs_accuracy.png")

# plot 2: parallel
fig, ax = plt.subplots(figsize=(8, 5))
ax.set_title("Parallel scaling — thinking tokens vs accuracy (AIME-2024)",
             fontsize=12, fontweight="bold")

ax.plot(par_think_toks, par_majority, "o-",  color=PAR_MAJ_COLOR,
        lw=2, markersize=7, label="majority_vote")
ax.plot(par_think_toks, par_best, "s--", color=PAR_BEST_COLOR,
        lw=2, markersize=7, label="best_of_n (oracle)")

for xi, ym, yb, n in zip(par_think_toks, par_majority, par_best, par_n):
    ax.annotate(f"n={n}\n{ym:.2f}", (xi, ym), textcoords="offset points",
                xytext=(0, 9), ha="center", fontsize=8, color=PAR_MAJ_COLOR)

ax.set_xlabel("Total thinking tokens (30 problems × n samples)")
ax.set_ylabel("Accuracy")
ax.xaxis.set_major_formatter(fmt)
ax.set_ylim(-0.02, 1.0)
ax.legend()
plt.tight_layout()
plt.savefig("par_thinking_tokens_vs_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: par_thinking_tokens_vs_accuracy.png")

# plot 3: exact vs flexible
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
fig.suptitle("Sequential vs Parallel — exact and flexible accuracy",
             fontsize=13, fontweight="bold")

for ax, mode, seq_acc, label in [
    (ax1, "exact_match", seq_exact, "exact_match"),
    (ax2, "flexible_extract", seq_flex, "flexible_extract"),
]:
    ax.plot(seq_think_toks, seq_acc,      "o-",  color=SEQ_EXACT_COLOR,
            lw=2, markersize=6, label="sequential")
    ax.plot(par_think_toks, par_majority, "o-",  color=PAR_MAJ_COLOR,
            lw=2, markersize=6, label="parallel majority")
    ax.plot(par_think_toks, par_best, "s--", color=PAR_BEST_COLOR,
            lw=2, markersize=6, label="parallel best-of-n")
    ax.set_title(f"eval mode: {label}")
    ax.set_xlabel("Total thinking tokens")
    ax.set_ylabel("Accuracy")
    ax.xaxis.set_major_formatter(fmt)
    ax.set_ylim(-0.02, 1.0)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("exact_vs_flexible_side_by_side.png", dpi=150, bbox_inches="tight")
plt.show()

# plot 4: avg total tokens per problem vs accuracy
# note: total = think + answer tokens
AVG_ANSWER_TOKS = 300

seq_avg_total = [t + AVG_ANSWER_TOKS for t in seq_avg_think]
par_avg_total = [t * n + AVG_ANSWER_TOKS * n
                 for t, n in zip(par_avg_think, par_n)]  # total per problem

fig, ax = plt.subplots(figsize=(9, 5))
ax.set_title("Avg total tokens per problem vs accuracy",
             fontsize=12, fontweight="bold")

ax.plot(seq_avg_total, seq_exact, "o-", color=SEQ_EXACT_COLOR, lw=2,
        markersize=7, label="sequential exact")
ax.plot(seq_avg_total, seq_flex, "s--", color=SEQ_FLEX_COLOR,  lw=2,
        markersize=7, label="sequential flexible")
ax.plot(par_avg_total, par_majority, "o-",  color=PAR_MAJ_COLOR,  lw=2,
        markersize=7, label="parallel majority")
ax.plot(par_avg_total, par_best, "s--", color=PAR_BEST_COLOR, lw=2,
        markersize=7, label="parallel best-of-n")

ax.set_xlabel("Avg total tokens per problem (think + answer)\n")
ax.set_ylabel("Accuracy")
ax.xaxis.set_major_formatter(fmt)
ax.set_ylim(-0.02, 1.0)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("total_tokens_vs_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

## Reasoning traces for part 2.3

In [ ]:
import os, json
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

from run_eval import *

TRACE_DIR = "traces"
os.makedirs(TRACE_DIR, exist_ok=True)
BATCH = 8

def run_sequential_batched(budget, problem_indices, label=""):
    print(f"\nsequential budget={budget} | {len(problem_indices)} problems [{label}]")
    params_think  = SamplingParams(
        max_tokens=budget, temperature=0.0,
        stop=["</think>"], include_stop_str_in_output=False,
    )
    params_answer = SamplingParams(max_tokens=2048, temperature=0.0)
    records = []

    for start in range(0, len(problem_indices), BATCH):
        batch_idx = problem_indices[start: start + BATCH]
        batch = [dataset[i] for i in batch_idx]
        labels = [int(ex["label"]) for ex in batch]
        prompts = [build_prompt(ex, enable_thinking=True) for ex in batch]

        think_outs = llm.generate(prompts, params_think)
        answer_prompts = []
        think_meta = []
        for j, out in enumerate(think_outs):
            think_text = out.outputs[0].text
            think_meta.append({
                "text": think_text,
                "tokens": len(out.outputs[0].token_ids),
                "finish": out.outputs[0].finish_reason,
            })
            answer_prompts.append(prompts[j] + think_text + "</think>\n")

        ans_outs = llm.generate(answer_prompts, params_answer)
        for j, ans_out in enumerate(ans_outs):
            answer_text = ans_out.outputs[0].text
            full_text = answer_prompts[j] + answer_text
            i = batch_idx[j]
            tm = think_meta[j]
            record = {
                "problem_idx": i,
                "problem": str(dataset[i]["prompt"])[:500],
                "gold": labels[j],
                "budget": budget,
                "think_tokens": tm["tokens"],
                "finish_reason": tm["finish"],
                "thinking_trace": tm["text"],
                "answer_text": answer_text,
                "pred_exact": extract_answer(full_text, "exact_match"),
                "pred_flex": extract_answer(full_text, "flexible_extract"),
            }
            record["correct_exact"] = record["pred_exact"] == labels[j]
            records.append(record)

    path = os.path.join(TRACE_DIR, f"seq_b{budget}_{label}.json")
    with open(path, "w") as f:
        json.dump(records, f, indent=2)
    return records

traces_low = run_sequential_batched(4000, list(range(30)), label="all")

low_correct = {r["problem_idx"] for r in traces_low if r["correct_exact"]}
low_incorrect = {r["problem_idx"] for r in traces_low if not r["correct_exact"]}

q1_candidates = sorted(low_incorrect)[:2]
q2_candidates = sorted(low_correct)[:2]
high_targets = sorted(set(q1_candidates + q2_candidates))

print(f"\ntargets (4 problems): {high_targets}")
traces_high = run_sequential_batched(16000, high_targets, label="targeted")

high_correct = {r["problem_idx"] for r in traces_high if r["correct_exact"]}
high_wrong = {r["problem_idx"] for r in traces_high if not r["correct_exact"]}

q1 = high_correct & low_incorrect
print(f"\nQ1: {sorted(q1)}")
for idx in sorted(q1)[:2]:
    r_lo = next(r for r in traces_low  if r["problem_idx"] == idx)
    r_hi = next(r for r in traces_high if r["problem_idx"] == idx)

q2 = low_correct & high_wrong
print(f"\nQ2: {sorted(q2)}")
for idx in sorted(q2)[:2]:
    r_lo = next(r for r in traces_low  if r["problem_idx"] == idx)
    r_hi = next(r for r in traces_high if r["problem_idx"] == idx)

## Improving parallel scaling (part 2.4)

In [ ]:
import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# evaluation points
TOTAL_BUDGETS = [16000, 32000]
PARALLEL_THINK_BUDGET = 4000   # per sample, same as baseline
ANSWER_TOKENS = 2048

# baseline results from 2.2
baseline = {
    16_000: {"majority": 0.2667, "best": 0.2667, "total_toks": 696547}, # n=4
    32_000: {"majority": 0.4000, "best": 0.4000, "total_toks": 1392257}, # n=8
}

# experiment 1: higher temperature (fix zero-diversity problem)
# motivation: majority==best_of_n in baseline means all samples
# are identical
def run_parallel_high_temp(total_budget):
    n_samples = total_budget // PARALLEL_THINK_BUDGET
    print(f"\nexperiment 1: high temp | total={total_budget} n={n_samples}")

    params = SamplingParams(
        max_tokens=PARALLEL_THINK_BUDGET + ANSWER_TOKENS,
        temperature=1.0,   # raised from 0.6
        top_p=0.95,
        top_k=50,
        n=n_samples,
    )

    correct_majority, correct_best, total, total_think_tokens = 0, 0, 0, 0
    per_problem_tokens = []

    for i in range(0, len(dataset), BATCH_SIZE):
        batch   = dataset.select(range(i, min(i + BATCH_SIZE, len(dataset))))
        labels  = [int(ex["label"]) for ex in batch]
        prompts = [build_prompt(ex, enable_thinking=True) for ex in batch]
        outputs = llm.generate(prompts, params)

        for j, out in enumerate(outputs):
            gold = labels[j]
            preds_exact = []
            problem_total = 0

            for sample in out.outputs:
                think_text, ans_text = extract_thinking_and_answer(sample.text)
                problem_total += count_tokens(think_text, tokenizer)
                problem_total += count_tokens(ans_text,   tokenizer)
                total_think_tokens += count_tokens(think_text, tokenizer)
                preds_exact.append(extract_answer(sample.text, "exact_match"))

            per_problem_tokens.append(problem_total)
            if majority_vote(preds_exact) == gold: correct_majority += 1
            if any(p == gold for p in preds_exact): correct_best   += 1
            total += 1

    acc_maj = correct_majority / total
    acc_best = correct_best / total
    print(f"majority={acc_maj:.4f} | best_of_{n_samples}={acc_best:.4f} | "
          f"avg_think_tokens={total_think_tokens/total:.1f}")
    return acc_maj, acc_best, total_think_tokens, per_problem_tokens


# experiment 2: hybrid, sequential think once, sample n answers
# motivation: spend half the budget on one deep WAIT chain to
# get high-quality reasoning, then sample n answers from it
# combines sequential depth with parallel answer diversity
def run_hybrid(total_budget, n_answers=4):
    """
    Split budget evenly:
      - Half on one sequential WAIT thinking chain
      - Half divided across n_answers parallel answer samples
    """
    think_budget = total_budget // 2
    print(f"\nexperiment 2: hybrid | total={total_budget} "
          f"think_budget={think_budget} n_answers={n_answers}")

    params_answer = SamplingParams(
        max_tokens=ANSWER_TOKENS,
        temperature=0.6,
        top_p=0.95,
        top_k=50,
        n=n_answers, # sample n answers from same thinking context
    )

    correct_majority, correct_best, total, total_think_tokens = 0, 0, 0, 0
    per_problem_tokens = []

    for i in range(0, len(dataset), BATCH_SIZE):
        batch  = dataset.select(range(i, min(i + BATCH_SIZE, len(dataset))))
        labels = [int(ex["label"]) for ex in batch]

        for j, ex in enumerate(batch):
            # one deep WAIT thinking chain
            current_prompt = build_prompt(ex, enable_thinking=True)
            think_tokens_used = 0

            while think_tokens_used < think_budget:
                remaining = think_budget - think_tokens_used
                params_think = SamplingParams(
                    max_tokens=remaining,
                    temperature=0.0,   # greedy for thinking chain
                    stop=["</think>"],
                    include_stop_str_in_output=False,
                )
                out = llm.generate([current_prompt], params_think)[0]
                chunk = out.outputs[0].text
                think_tokens_used += len(out.outputs[0].token_ids)

                if out.outputs[0].finish_reason == "stop" and think_tokens_used < think_budget:
                    current_prompt += chunk + "Wait\n"
                else:
                    current_prompt += chunk
                    break

            total_think_tokens += think_tokens_used

            # sample n answers
            answer_prompt = current_prompt + "</think>\n"
            ans_out = llm.generate([answer_prompt], params_answer)[0]

            preds_exact = []
            problem_total = think_tokens_used

            for sample in ans_out.outputs:
                problem_total += len(sample.token_ids)
                full_text = answer_prompt + sample.text
                preds_exact.append(extract_answer(full_text, "exact_match"))

            per_problem_tokens.append(problem_total)
            gold = labels[j]
            if majority_vote(preds_exact) == gold: correct_majority += 1
            if any(p == gold for p in preds_exact): correct_best   += 1
            total += 1

    acc_maj  = correct_majority / total
    acc_best = correct_best / total
    print(f"  majority={acc_maj:.4f} | best_of_{n_answers}={acc_best:.4f} | "
          f"avg_think_tokens={total_think_tokens/total:.1f}")
    return acc_maj, acc_best, total_think_tokens, per_problem_tokens


# both stategies at both budget points
s1_results = {}   # high temp
s2_results = {}   # hybrid

for budget in TOTAL_BUDGETS:
    s1_results[budget] = run_parallel_high_temp(budget)
    s2_results[budget] = run_hybrid(budget, n_answers=4)

budgets = TOTAL_BUDGETS
x = np.arange(len(budgets))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
ax.set_title("Improved parallel scaling at fixed compute budgets",
             fontsize=12, fontweight="bold")

#mMajority vote accuracy at each budget point
baseline_acc = [baseline[b]["majority"] for b in budgets]
s1_acc  = [s1_results[b][0] for b in budgets]
s2_acc = [s2_results[b][0] for b in budgets]

ax.bar(x - width, baseline_acc, width, label="Baseline (temp=0.6)",  color="#93c5fd")
ax.bar(x, s1_acc, width, label="Strategy 1 (temp=1.0)", color="#2563eb")
ax.bar(x + width, s2_acc, width, label="Strategy 2 (hybrid)",   color="#16a34a")

for bars in [baseline_acc, s1_acc, s2_acc]:
    for xi, v, offset in zip(x, bars,
                              [x - width, x, x + width]):
        pass

for idx, (vals, offset) in enumerate(zip(
    [baseline_acc, s1_acc, s2_acc],
    [-width, 0, width]
)):
    for xi, v in zip(x, vals):
        ax.text(xi + offset, v + 0.01, f"{v:.2f}",
                ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([f"{b//1000}K tokens" for b in budgets])
ax.set_ylabel("Majority vote accuracy")
ax.set_ylim(0, 1.0)
ax.legend()
ax.grid(axis="y", alpha=0.3, linestyle="--")

plt.tight_layout()
plt.savefig("part_2_4_experiments.png", dpi=150, bbox_inches="tight")
plt.show()

#summary
print("\nsummary for part 2.4")
print(f"{'Strategy':<25} {'16K acc':>8} {'32K acc':>8}")
print(f"{'Baseline (temp=0.6)':<25} {baseline[16000]['majority']:>8.4f} "
      f"{baseline[32000]['majority']:>8.4f}")
print(f"{'Strategy 1 (temp=1.0)':<25} {s1_results[16000][0]:>8.4f} "
      f"{s1_results[32000][0]:>8.4f}")
print(f"{'Strategy 2 (hybrid)':<25} {s2_results[16000][0]:>8.4f} "
      f"{s2_results[32000][0]:>8.4f}")

Plot part 2.4

In [ ]:
# hardcoded results

TOTAL_BUDGETS = [16_000, 32_000]

baseline = {
    16_000: {"majority": 0.2667, "total_toks": 696_547},
    32_000: {"majority": 0.4000, "total_toks": 1_392_257},
}

# strategy 1: high temperature
s1 = {
    16_000: {"majority": 0.2667, "best": 0.2667},
    32_000: {"majority": 0.3667, "best": 0.4000},
}

# strategy 2: hybrid
s2 = {
    16_000: {"majority": 0.6000, "best": 0.6000, "measured": True},
    32_000: {"majority": 0.65,   "best": 0.65,   "measured": False},  # estimated
}

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": "--",
})

BASE_COLOR = "#6b7280"
S1_COLOR = "#2563eb"
S2_COLOR = "#16a34a"

x = np.array(TOTAL_BUDGETS)
x_labs = ["16K tokens", "32K tokens"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.set_title("Section 2.4 — Improved parallel scaling at fixed compute budgets\n"
             "(Qwen3-4B, AIME-2024, majority vote accuracy)",
             fontsize=11, fontweight="bold")

base_acc = [baseline[b]["majority"] for b in TOTAL_BUDGETS]
s1_acc = [s1[b]["majority"] for b in TOTAL_BUDGETS]
s2_acc = [s2[b]["majority"] for b in TOTAL_BUDGETS]

ax.plot(x, base_acc, "o-", color=BASE_COLOR, lw=2, markersize=8,
        label="Baseline (T=0.6, parallel)", zorder=3)

ax.plot(x, s1_acc, "s-", color=S1_COLOR, lw=2, markersize=8,
        label="Strategy 1: high temp (T=1.0)", zorder=3)

s2_measured = [b for b in TOTAL_BUDGETS if s2[b]["measured"]]
s2_estimated = [b for b in TOTAL_BUDGETS if not s2[b]["measured"]]
s2_acc_meas = [s2[b]["majority"] for b in s2_measured]
s2_acc_est = [s2[b]["majority"] for b in s2_estimated]

ax.plot([b for b in s2_measured], s2_acc_meas,
        "^", color=S2_COLOR, markersize=9, zorder=4,
        label="Strategy 2: hybrid seq+par (measured)")
if s2_estimated:
    ax.plot([b for b in s2_estimated], s2_acc_est,
            "^", color=S2_COLOR, markersize=9,
            markerfacecolor="white", markeredgewidth=2, zorder=4,
            label="Strategy 2: hybrid seq+par (estimated)")
    ax.plot(x, s2_acc, "--", color=S2_COLOR, lw=2, zorder=2)
else:
    ax.plot(x, s2_acc, "-", color=S2_COLOR, lw=2, zorder=2)

for xi, yb, y1, y2 in zip(x, base_acc, s1_acc, s2_acc):
    ax.annotate(f"{yb:.2f}", (xi, yb), textcoords="offset points",
                xytext=(-18, 6), fontsize=9, color=BASE_COLOR)
    ax.annotate(f"{y1:.2f}", (xi, y1), textcoords="offset points",
                xytext=(6, 6), fontsize=9, color=S1_COLOR)
    ax.annotate(f"{y2:.2f}{'*' if not s2[xi]['measured'] else ''}",
                (xi, y2), textcoords="offset points",
                xytext=(6, -14), fontsize=9, color=S2_COLOR)

ax.set_xticks(x)
ax.set_xticklabels(x_labs)
ax.set_ylabel("Majority vote accuracy (exact_match)")
ax.set_ylim(0, 0.85)
ax.legend(loc="upper left", fontsize=9)
ax.text(0.98, 0.04, "* estimated (run incomplete)",
        transform=ax.transAxes, ha="right", fontsize=8, color="gray",
        style="italic")

plt.tight_layout()
plt.savefig("section_2_4_overlay.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ax.set_title("Section 2.4 — Strategy comparison at 16K and 32K token budgets",
             fontsize=11, fontweight="bold")

width = 0.22
x_pos = np.array([0, 1])

bars_base = ax.bar(x_pos - width, base_acc, width,
                   color=BASE_COLOR, alpha=0.85, label="Baseline (T=0.6)")
bars_s1 = ax.bar(x_pos,          s1_acc,   width,
                   color=S1_COLOR,  alpha=0.85, label="Strategy 1 (T=1.0)")
bars_s2 = ax.bar(x_pos + width,  s2_acc,   width,
                   color=S2_COLOR,  alpha=0.85, label="Strategy 2 (hybrid)")

for i, b in enumerate(TOTAL_BUDGETS):
    if not s2[b]["measured"]:
        bars_s2[i].set_hatch("///")
        bars_s2[i].set_alpha(0.5)

for bars in [bars_base, bars_s1, bars_s2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                f"{h:.2f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x_pos)
ax.set_xticklabels(x_labs)
ax.set_ylabel("Majority vote accuracy")
ax.set_ylim(0, 0.85)
ax.legend(fontsize=9)
ax.text(0.98, 0.04, "Hatched bar = estimated (run incomplete)",
        transform=ax.transAxes, ha="right", fontsize=8,
        color="gray", style="italic")

plt.tight_layout()
plt.savefig("section_2_4_bars.png", dpi=150, bbox_inches="tight")
plt.show()